# Step 6 · 100-episode dreams on the Kaggle GPU (3 settings)

Settings: **Accelerator → GPU T4 x2**, **Internet → On**. **Add Input → Your Datasets**: the main dataset, the DAgger dataset and the checkpoints dataset (`world_model_v1.pt`, `world_model_v1_noaug.pt`). No W&B secret needed.

No rendering happens here: dreams start from stored frames. Download `step6_dreams.zip` afterwards; tracking and the action test run on the laptop.

In [ ]:
# 1. Fresh clone of the repo
%cd /kaggle/working
!rm -rf robot-world-model
!git clone -q https://github.com/Swayam-27/robot-world-model.git
%cd /kaggle/working/robot-world-model
!git log --oneline -1

In [ ]:
# 2. Find the two episode datasets and the two checkpoints
import glob, os
import torch
assert torch.cuda.is_available(), "turn the GPU on: Settings -> Accelerator -> GPU T4 x2"
hits = [os.path.dirname(h) for h in glob.glob("/kaggle/input/**/episodes.csv", recursive=True)]
main = [h for h in hits if "dagger" not in h.lower()]
dagger = [h for h in hits if "dagger" in h.lower()]
assert len(main) == 1 and len(dagger) == 1, f"attach both datasets: main {main}, dagger {dagger}"
DATA, DAGGER = main[0], dagger[0]
CKPT = {}
for name in ("world_model_v1", "world_model_v1_noaug"):
    found = [f for f in glob.glob(f"/kaggle/input/**/{name}.pt", recursive=True) if os.path.isfile(f)]
    assert len(found) == 1, f"expected one {name}.pt in the checkpoints dataset, found {found}"
    CKPT[name] = found[0]
print(DATA, DAGGER, CKPT, sep="\n")

In [ ]:
# 3. Dream 100 episodes per setting (own actions, then another brain's actions). Stops at the first error.
import subprocess, sys, yaml
for cfg in ("dream_v1", "dream_v1_aug03", "dream_v1_noaug"):
    wm = yaml.safe_load(open(f"configs/{cfg}.yaml"))["world_model"]
    print("=====", cfg, flush=True)
    subprocess.run([sys.executable, "-u", "-m", "eval.dream", "--config", f"configs/{cfg}.yaml", "--checkpoint", CKPT[wm],
                    "--data-dir", DATA, "--data-dir", DAGGER], check=True)

In [ ]:
# 4. Pack the results (download step6_dreams.zip from the Output panel, unzip into outputs/)
!cd outputs && zip -qr /kaggle/working/step6_dreams.zip dream_v1 dream_v1_aug03 dream_v1_noaug && ls -lh /kaggle/working/step6_dreams.zip